### Simulation log code
This is a simulation log code snippet can be parametrized with the goal of saving on a csv file the simulation data at each time step.

In [37]:
from main import *
from Utils.plotting_functions import *
from Worlds.World import World
import Drone.Simulation
from matplotlib import pyplot as plt
from Optimizations.PSO_optimizer import PSOOptimizer
from Optimizations.optimizer import Optimizer as CostWrapper
import os

parameters = load_parameters("Settings/simulation_parameters.yaml")
world = World.load_world(parameters['world_data_path'])

Set turbolence flag to automatically activate turbolence in the simulation

In [38]:
TURBOLENCE_FLAG = False  # Set to True to enable turbulence in the simulations
WIND_HEIGHT = 100
WIND_AIRSPEED = 10
TURBOLENCE_LEVEL = 30

### Set simulation waypoints
A and B are the starting and ending point. The vector waypoints define the trajectory waypoints. Change them to chage drone's trajectory. "v" key defines the target velocity in m/s that is set the selected waypoint

In [39]:
A = parameters['start_point']
B = parameters['end_point']

waypoints = [
    {"x": 27.664927049116653, "y": 88.29509735107422, "z": 10.923457145690918, "v": 18.322269439697266}, 
    {"x": 37.11072626980868, "y": 85.373291015625, "z": 23.00494384765625, "v": 13.508941650390625}, 
    {"x": 74.18110830133611, "y": 89.29581451416016, "z": 26.83894920349121, "v": 15.422500610351562}, 
    {"x": 95.0, "y": 50.0, "z": 1.0, "v": 15.422500610351562}
]

Load default PID values (saved in parameter yaml file) and create controller and drone's object

In [ ]:

init_state = create_initial_state(A[0], A[1], A[2])
thrust_max = get_max_thrust_from_rotor_model(parameters)

k_pid_pos = tuple(map(float, parameters['k_pid_pos']))
k_pid_alt = tuple(map(float, parameters['k_pid_alt']))
k_pid_att = tuple(map(float, parameters['k_pid_att']))
k_pid_yaw = tuple(map(float, parameters['k_pid_yaw']))
k_pid_hsp = tuple(map(float, parameters['k_pid_hsp']))
k_pid_vsp = tuple(map(float, parameters['k_pid_vsp']))

all_pid_list = [*k_pid_pos, *k_pid_alt, *k_pid_att, *k_pid_yaw, *k_pid_hsp, *k_pid_vsp]

def build_pid_gains(pid_list):
    return {
            'k_pid_pos': (pid_list[0], pid_list[1], pid_list[2]),
            'k_pid_alt': (pid_list[3], pid_list[4], pid_list[5]),
            'k_pid_att': (pid_list[6], pid_list[7], pid_list[8]),
            'k_pid_yaw': (pid_list[9], pid_list[10], pid_list[11]),
            'k_pid_hsp': (pid_list[12], pid_list[13], pid_list[14]),
            'k_pid_vsp': (pid_list[15], pid_list[16], pid_list[17]),
    }

pid_gains = build_pid_gains(all_pid_list)
quad_controller = create_quadcopter_controller(init_state, pid_gains, thrust_max, parameters)
drone = create_quadcopter_model(init_state, quad_controller, parameters)
# noise_model = load_dnn_noise_model(parameters)

Creating the pandas dataframe and first simulation settings

In [ ]:
import pandas as pd

experiment_dataset = pd.DataFrame(
    columns=[
        'varied_pid_name',
        'k_pid_pos', 
        'k_pid_alt', 
        'k_pid_att', 
        'k_pid_yaw', 
        'k_pid_hsp', 
        'k_pid_vsp', 
        'time_history',
        'position_history',
        'rpm_history',
        'pitch_roll_yaw_history',
        'vertical_speed_history',
        'horizontal_speed_history',
    ]
)


sim_1 = Simulation(
    drone,
    world,
    waypoints,
    dt=float(parameters['dt']),
    max_simulation_time=float(parameters['simulation_time']),
    frame_skip=int(parameters['frame_skip']),
    target_reached_threshold=float(parameters['threshold']),
    target_shift_threshold_distance=float(parameters['target_shift_threshold_distance']),
    noise_model=None,
    generate_sound_emission_map=True,
    compute_psychoacoustics=False,
    noise_annoyance_radius=0,
)
if TURBOLENCE_FLAG: sim_1.setWind(
    height=WIND_HEIGHT,
    airspeed=WIND_AIRSPEED,
    turbulence_level=TURBOLENCE_LEVEL,
    plot_wind_signal=True
)

Save in the first row of the dataset the first log data with no PID value perturbation

In [42]:
sim_1.startSimulation()
experiment_dataset = pd.concat([experiment_dataset, pd.DataFrame({
    'varied_pid_name': [None],
    'k_pid_pos': [pid_gains['k_pid_pos']],
    'k_pid_alt': [pid_gains['k_pid_alt']],
    'k_pid_att': [pid_gains['k_pid_att']],
    'k_pid_yaw': [pid_gains['k_pid_yaw']],
    'k_pid_hsp': [pid_gains['k_pid_hsp']],
    'k_pid_vsp': [pid_gains['k_pid_vsp']],
    'time_history': [sim_1.time_history],
    'position_history': [sim_1.positions],
    'rpm_history': [sim_1.rpms_history],
    'pitch_roll_yaw_history': [sim_1.angles_history],
    'vertical_speed_history': [sim_1.vertical_speed_history],
    'horizontal_speed_history': [sim_1.horiz_speed_history],
}, index=[0])], ignore_index=True)

Final target reached at time: 17.79 s
Simulation completed in 4.19 seconds.


### Apply the perturbations to each p - i - d value
Cycling across all the PID values applying a percentage variation defined in the variable "prc_var".

In [ ]:
from tqdm import tqdm

prc_var = 0.01

for i in tqdm(range(len(all_pid_list)), total=len(all_pid_list)):
    perturbed_pid_list = all_pid_list.copy()

    perturbation = all_pid_list[i] * prc_var
    perturbed_pid_list[i] += perturbation

    pid_gains = build_pid_gains(perturbed_pid_list)
    quad_controller = create_quadcopter_controller(init_state, pid_gains, thrust_max, parameters)
    drone = create_quadcopter_model(init_state, quad_controller, parameters)
    sim = Simulation(
        drone,
        world,
        waypoints,
        dt=float(parameters['dt']),
        max_simulation_time=float(parameters['simulation_time']),
        frame_skip=int(parameters['frame_skip']),
        target_reached_threshold=float(parameters['threshold']),
        target_shift_threshold_distance=float(parameters['target_shift_threshold_distance']),
        noise_model=None,
        generate_sound_emission_map=True,
        compute_psychoacoustics=False,
        noise_annoyance_radius=0,
    )
    sim.startSimulation(verbose=False)
    if TURBOLENCE_FLAG: sim.setWind(
        height=WIND_HEIGHT,
        airspeed=WIND_AIRSPEED,
        turbulence_level=TURBOLENCE_LEVEL,
        plot_wind_signal=True
    )

    pid_set_name = list(pid_gains.keys())[i // 3]
    pid_term = 'P' if i % 3 == 0 else ('I' if i % 3 == 1 else 'D')

    experiment_dataset = pd.concat([experiment_dataset, pd.DataFrame({   
            'varied_pid_name': [pid_set_name + '_' + pid_term],
            'k_pid_pos': [pid_gains['k_pid_pos']], 
            'k_pid_alt': [pid_gains['k_pid_alt']],
            'k_pid_att': [pid_gains['k_pid_att']],
            'k_pid_yaw': [pid_gains['k_pid_yaw']],
            'k_pid_hsp': [pid_gains['k_pid_hsp']],
            'k_pid_vsp': [pid_gains['k_pid_vsp']],
            'time_history': [sim.time_history],
            'position_history': [sim.positions],
            'rpm_history': [sim.rpms_history],
            'pitch_roll_yaw_history': [sim.angles_history],
            'vertical_speed_history': [sim.vertical_speed_history],
            'horizontal_speed_history': [sim.horiz_speed_history],

        }
    )], ignore_index=True)

 44%|████▍     | 8/18 [00:38<00:47,  4.78s/it]


KeyboardInterrupt: 

In [ ]:
# Save the dataset to a CSV file separated by semicolons

# Create outputs directory if it doesn't exist
output_folder_path = 'Outputs'

if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

experiment_dataset.to_csv(f'{output_folder_path}/pid_gains_sensitivity_analysis_dataset{"_turbulence" if TURBOLENCE_FLAG else ""}.csv', sep=';', index=False)
display(experiment_dataset)

# When loading the histories, use ast ast.literal_eval to convert the string representation of lists back to actual lists
# Example:
# import ast
# df = pd.read_csv('path_to_csv_file.csv', sep=';')
# df['time_history'] = df['time_history'].apply(ast.literal_eval)

,varied_pid_name,k_pid_pos,k_pid_alt,k_pid_att,k_pid_yaw,k_pid_hsp,k_pid_vsp,time_history,position_history,rpm_history,pitch_roll_yaw_history,vertical_speed_history,horizontal_speed_history
0,None,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.08, 0.16, 0.24, 0.32, 0.4, 0.48, 0.56,...","[[5.000009482480716, 50.00000949729613, 1.0001...","[[0.0, 3000.0, 3000.0, 0.0], [2275.82200792443...","[[-0.0690763502481655, 0.0690763502481655, 0.0...","[0.048896166614490304, 0.778995798869476, 1.36...","[0.0017774386640726275, 0.2951429896184142, 1...."
1,k_pid_pos_P,"(1.128812721359806, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.08, 0.16, 0.24, 0.32, 0.4, 0.48, 0.56,...","[[5.000009482480716, 50.00000949729613, 1.0001...","[[0.0, 3000.0, 3000.0, 0.0], [2275.82200792443...","[[-0.0690763502481655, 0.0690763502481655, 0.0...","[0.048896166614490304, 0.778995798869476, 1.36...","[0.0017774386640726275, 0.2951429896184142, 1...."
2,k_pid_pos_I,"(1.026193383054369, 0.0369497837668771, 0.3053...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.08, 0.16, 0.24, 0.32, 0.4, 0.48, 0.56,...","[[5.000009482480716, 50.00000949729613, 1.0001...","[[0.0, 3000.0, 3000.0, 0.0], [2275.82200792443...","[[-0.0690763502481655, 0.0690763502481655, 0.0...","[0.048896166614490304, 0.778995798869476, 1.36...","[0.0017774386640726275, 0.2951429896184142, 1...."
3,k_pid_pos_D,"(1.026193383054369, 0.03359071251534282, 0.335...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.08, 0.16, 0.24, 0.32, 0.4, 0.48, 0.56,...","[[5.000009482480716, 50.00000949729613, 1.0001...","[[0.0, 3000.0, 3000.0, 0.0], [2275.82200792443...","[[-0.0690763502481655, 0.0690763502481655, 0.0...","[0.048896166614490304, 0.778995798869476, 1.36...","[0.0017774386640726275, 0.2951429896184142, 1...."
4,k_pid_alt_P,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.8014958751426673, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.08, 0.16, 0.24, 0.32, 0.4, 0.48, 0.56,...","[[5.000009482480716, 50.00000949729613, 1.0001...","[[0.0, 3000.0, 3000.0, 0.0], [2275.82200792443...","[[-0.0690763502481655, 0.0690763502481655, 0.0...","[0.048896166614490304, 0.778995798869476, 1.36...","[0.0017774386640726275, 0.2951429896184142, 1...."
5,k_pid_alt_I,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.08, 0.16, 0.24, 0.32, 0.4, 0.48, 0.56,...","[[5.000009482480716, 50.00000949729613, 1.0001...","[[0.0, 3000.0, 3000.0, 0.0], [2275.82200792443...","[[-0.0690763502481655, 0.0690763502481655, 0.0...","[0.048896166614490304, 0.778995798869476, 1.36...","[0.0017774386640726275, 0.2951429896184142, 1...."
6,k_pid_alt_D,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.022969196847979254)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.2175139